# Correlation matrices + rates/asset scatter

1. **Level corr** — `CORR_SERIES`
2. **Corr percentile** — 100 = más alto en `CORR_PCT_LOOKBACK`
3. **Scatter** — x = ρ(USGG5YR, CL1), y = ρ(USGG5YR, ESA); últimos 4 días en azul
4. **Excel export** — `exports/correlations/corr_panel.xlsx` (levels + meta) para rolling corrs en Excel

Tickers que falten (`CL1`, `USSFCT*`, …) se intentan bajar solos de BBG.
Si `CL1`/`CO1` falla con `NOT_ENTITLED`, el scatter usa SPX o se salta.


In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "correlations.py").exists() else _DZ
if not (REPO / "correlations.py").exists():
    raise FileNotFoundError("No encuentro correlations.py — git pull origin main")

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())

In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
import curve_config
import correlations
import data

importlib.reload(curve_config)
importlib.reload(data)
importlib.reload(correlations)

print("CORR_SERIES:", curve_config.CORR_SERIES)
print("window=%s  pct_lookback=%s" % (
    curve_config.CORR_WINDOW, curve_config.CORR_PCT_LOOKBACK,
))

In [ ]:
level, pmat, full = correlations.run_correlations()
print("panel cols sample:", list(full.columns)[:20], "...")
print("asof:", full.index[-1], "n=", len(full))
print("USSFCT:", [c for c in full.columns if "USSFCT" in str(c)])
print("Excel export:", correlations.LAST_EXPORT_PATH)


In [ ]:
display(level.round(3))

In [ ]:
display(pmat.round(1))
heat = Path(curve_config.GRAPH_DIR_CORR) / "corr_percentile_heatmap.png"
display(Image(filename=str(heat))) if heat.exists() else print("no heatmap")

In [ ]:
heat_l = Path(curve_config.GRAPH_DIR_CORR) / "corr_level_heatmap.png"
display(Image(filename=str(heat_l))) if heat_l.exists() else None

In [ ]:
heat_s = Path(curve_config.GRAPH_DIR_CORR) / "corr_scatter_rates_assets.png"
if heat_s.exists():
    display(Image(filename=str(heat_s)))
elif scatter_path and Path(scatter_path).exists():
    display(Image(filename=str(scatter_path)))
else:
    print("no scatter (often CO1 NOT_ENTITLED — sin permiso BBG)")
if scatter_df is not None:
    display(scatter_df.tail(10))